# Tutorial 12: Empirical ML System Diagnostics, Learning Curves &

Regularization

Mathematics for Machine Learning (Mathe3)

Thomas Manke  
2026-10-07

## 1. Environment Setup (5 min)

Run the environment check cell below to confirm Python 3, NumPy, SciPy,
Matplotlib, Seaborn, and Scikit-Learn are available:

``` {python}
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")
print("Environment check passed.")
print(f"NumPy Version: {np.__version__}")
```

------------------------------------------------------------------------

## 2. Task 1: Team Model Setup — High Bias vs. High Variance (15 min)

In this exercise, we evaluate patient health data predicting **Systolic
Blood Pressure ($y$, in $\text{mmHg}$)** from **Body Mass Index ($x$, in
$\text{kg/m}^2$)**.

Each team is assigned a model with a deliberate structural failure mode:

- **Team A (High Bias Model Setup)**:
  - You are assigned a simple 1st-degree Linear Model
    ($\hat{y} = w_0 + w_1 x$). Because true physiology is non-linear,
    your model lacks structural capacity (**Underfitting / High Bias**).
- **Team B (High Variance Model Setup)**:
  - You are assigned an un-regularized 10th-degree Polynomial Model
    ($\hat{y} = \sum_{j=0}^{10} w_j x^j$). Your model has excessive
    capacity and fits patient measurement noise (**Overfitting / High
    Variance**).

``` {python}
# Task 1 Code Framework: Run for your Team's Assigned Model
np.random.seed(42)

# Generate synthetic patient dataset: Systolic BP (mmHg) vs. BMI (kg/m^2)
n_total = 500
x_all = np.random.uniform(18, 35, n_total)
y_all = 110 + 1.2 * x_all + 0.05 * (x_all - 25)**2 + np.random.normal(0, 4.0, n_total)

# Initial small training set (m = 25) and validation set (n_val = 200)
x_train_small, y_train_small = x_all[:25], y_all[:25]
x_val, y_val = x_all[100:300], y_all[100:300]

# --- UNCOMMENT YOUR TEAM'S ASSIGNED MODEL BELOW ---
# TEAM = 'A'  # Team A: Degree 1 Linear Regression (High Bias)
TEAM = 'B'    # Team B: Degree 10 Polynomial Regression (High Variance)

deg = 1 if TEAM == 'A' else 10
model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
model.fit(x_train_small.reshape(-1, 1), y_train_small)

# Evaluate Mean Squared Error (MSE) in mmHg^2
train_mse = mean_squared_error(y_train_small, model.predict(x_train_small.reshape(-1, 1)))
val_mse = mean_squared_error(y_val, model.predict(x_val.reshape(-1, 1)))

print(f"--- TEAM {TEAM} INITIAL DIAGNOSTICS (Degree {deg}) ---")
print(f"Training Loss  J_train: {train_mse:.2f} mmHg^2")
print(f"Validation Loss J_cv:   {val_mse:.2f} mmHg^2")
print(f"Generalization Gap:     {abs(val_mse - train_mse):.2f} mmHg^2")
```

------------------------------------------------------------------------

## 3. Task 2: Empirical Learning Curves ($J_{\text{train}}$ vs. $J_{\text{cv}}$ vs. Sample Size $m$) (30 min)

Following Andrew Ng’s CS229 diagnostic workflow, we plot **Learning
Curves** by evaluating $J_{\text{train}}$ and $J_{\text{cv}}$ across
increasing training sample sizes $m \in \{10, 25, 50, 100, 200, 300\}$.

### Instructions:

1.  Execute the code cell below to plot your team’s Learning Curve ($m$
    vs. MSE).
2.  Observe whether your curves plateau early with a small gap (Team A)
    or maintain a large gap that slowly narrows (Team B).

``` {python}
# Task 2: Learning Curve Diagnostics
m_sizes = [10, 20, 35, 50, 100, 150, 200]
train_errors = []
val_errors = []

for m in m_sizes:
    x_tr, y_tr = x_all[:m], y_all[:m]
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(x_tr.reshape(-1, 1), y_tr)
    
    tr_err = mean_squared_error(y_tr, model.predict(x_tr.reshape(-1, 1)))
    v_err = mean_squared_error(y_val, model.predict(x_val.reshape(-1, 1)))
    
    train_errors.append(tr_err)
    val_errors.append(v_err)

plt.figure(figsize=(7, 4.5))
plt.plot(m_sizes, train_errors, 'b-o', lw=2, label='Training Loss $J_{\\text{train}}$')
plt.plot(m_sizes, val_errors, 'r-o', lw=2, label='Validation Loss $J_{\\text{cv}}$')
plt.title(f'Team {TEAM} (Degree {deg}): Learning Curve Diagnostics', fontsize=10, fontweight='bold')
plt.xlabel('Training Set Size $m$', fontsize=9)
plt.ylabel('Loss $J(\\boldsymbol{w})$ ($\text{mmHg}^2$)', fontsize=9)
plt.ylim(0, max(val_errors[:3]) + 10)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

> **Learning Curve Interpretation:**
>
> - **Team A (High Bias)**: As $m$ increases, $J_{\text{train}}$ and
>   $J_{\text{cv}}$ plateau at a high error level with a tiny gap.
>   *Collecting more training data will NOT help!*
> - **Team B (High Variance)**: $J_{\text{train}}$ remains very low
>   while $J_{\text{cv}}$ is high. A large gap exists but narrows as $m$
>   grows. *Collecting more training data WILL help!*

------------------------------------------------------------------------

## 4. Task 3: Regularization Parameter Sweeps ($\lambda$) & Andrew Ng’s 6 Fixes (30 min)

In this exercise, we apply L2 Ridge Regularization
($\lambda \|\boldsymbol{w}\|_2^2$) to control model capacity across
penalty hyperparameter $\lambda \in [10^{-4}, 10^3]$.

### Instructions:

1.  Run the regularization sweep below.
2.  Find the optimal penalty hyperparameter $\lambda^*$ that minimizes
    validation error $J_{\text{cv}}(\lambda)$.

``` {python}
# Task 3: Regularization Parameter Sweep (L2 Ridge)
alphas = np.logspace(-4, 3, 50)
reg_train_err = []
reg_val_err = []

for a in alphas:
    # Fit Degree 10 model with L2 Ridge penalty alpha
    reg_model = make_pipeline(PolynomialFeatures(10), Ridge(alpha=a))
    reg_model.fit(x_train_small.reshape(-1, 1), y_train_small)
    
    tr_err = mean_squared_error(y_train_small, reg_model.predict(x_train_small.reshape(-1, 1)))
    v_err = mean_squared_error(y_val, reg_model.predict(x_val.reshape(-1, 1)))
    
    reg_train_err.append(tr_err)
    reg_val_err.append(v_err)

best_idx = np.argmin(reg_val_err)
best_alpha = alphas[best_idx]

plt.figure(figsize=(7, 4.5))
plt.plot(alphas, reg_train_err, 'b-', lw=2, label='Training Loss $J_{\\text{train}}(\\lambda)$')
plt.plot(alphas, reg_val_err, 'r-', lw=2, label='Validation Loss $J_{\\text{cv}}(\\lambda)$')
plt.axvline(best_alpha, color='black', linestyle='--', label=f'Optimal $\\lambda^* = {best_alpha:.3f}$')
plt.xscale('log')
plt.title('Task 3: Regularization Sweep (L2 Ridge Penalty $\\lambda$)', fontsize=10, fontweight='bold')
plt.xlabel('Regularization Parameter $\\lambda$ (log scale)', fontsize=9)
plt.ylabel('Loss $J(\\boldsymbol{w})$ ($\text{mmHg}^2$)', fontsize=9)
plt.ylim(0, 50)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Optimal Regularization Parameter lambda*: {best_alpha:.4f}")
print(f"Minimum Validation Error J_cv(lambda*):  {reg_val_err[best_idx]:.2f} mmHg^2")
```

------------------------------------------------------------------------

## 5. Team Presentation & Diagnostic Decision Matrix (15 min)

Compare Team A’s results vs. Team B’s results using Andrew Ng’s **6
Fixes Decision Matrix**:

| Action / Fix | Solves High Bias (Team A)? | Solves High Variance (Team B)? |
|:--------------------|:------------------------:|:------------------------:|
| **Get more training data ($m$)** | ❌ **Useless** | ✅ **Fixes High Variance** |
| **Try a smaller set of features** | ❌ **Useless** | ✅ **Fixes High Variance** |
| **Try adding new features / polynomial terms** | ✅ **Fixes High Bias** | ❌ **Useless** |
| **Decrease regularization parameter $\lambda$** | ✅ **Fixes High Bias** | ❌ **Useless** |
| **Increase regularization parameter $\lambda$** | ❌ **Useless** | ✅ **Fixes High Variance** |

> **Final Diagnostic Reflection**
>
> - **Team A Recommendation**: Since your model suffers from High Bias,
>   you MUST increase model capacity (add non-linear features, increase
>   polynomial degree, or decrease $\lambda$).
> - **Team B Recommendation**: Since your model suffers from High
>   Variance, you MUST constrain model capacity (apply L2 Ridge
>   regularization $\lambda^* \approx 0.1$ or collect more patient
>   data).